In [1]:
import pandas as pd
import numpy as np
import json
import re
import time
import math
from pathlib import Path
import requests
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import unicodedata

pd.set_option("display.max_colwidth", 120)
pio.templates.default = "plotly_white"


In [2]:
panel = pd.read_csv("interviews.csv")
COMP_COLS = [f"Q_comparaison_{i}" for i in range(1, 7)]
df = panel[["panelist_id"] + COMP_COLS].copy()
print(panel.shape)
df.head(2)


(800, 48)


,panelist_id,Q_comparaison_1,Q_comparaison_2,Q_comparaison_3,Q_comparaison_4,Q_comparaison_5,Q_comparaison_6
0,67b0d135d362f5886c2e5cfe,"J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'...","Celle de Bouygues, sans hésiter. L'idée de la scène de crime pour du wifi, c'est tellement absurde que tu t'en souvi...","Bouygues. Clairement. Pour son concept. Ils ont osé faire un truc complètement différent, une parodie. Ça sort du ca...","Bouygues. Ça m'a fait rire. C'est une émotion simple, mais c'est positif. La pub Orange, elle ne suscite pas vraimen...","C'est marrant parce que même si j'ai préféré la pub Bouygues, je crois que c'est celle d'Orange qui me ferait le plu...","Je dirais Orange. Leur promesse c'est 'la fiabilité', et toute la pub est construite pour te montrer pourquoi c'est ..."
1,67b0d12ed362f5886c2e5b9c,Hmm... difficile. J'ai bien aimé les deux pour des raisons différentes. Mais je vais dire Bouygues. Juste pour l'ori...,"Celle de Bouygues, je pense. L'idée de la 'police du WiFi' c'est un concept fort, facile à retenir et à raconter. La...","Bouygues, sans aucune hésitation. Pour tout le côté parodie de série policière. C'est un vrai parti pris créatif. Or...","Bouygues m'a fait plus rire. Donc si on parle d'émotion forte, le rire, c'est celle-là. Orange, c'est plus une sympa...","Alors là, c'est marrant, mais je dirais peut-être Orange. Même si j'ai préféré la pub Bouygues. Parce que le message...","Je dirais Orange. Leur promesse c'est la fiabilité, et ils le montrent bien avec des exemples où tout le reste échou..."


In [3]:
def build_verbatim(row):
    parts = [str(row[col]) for col in COMP_COLS if pd.notna(row[col])]
    return " ".join(parts)

df["verbatim"] = df.apply(build_verbatim, axis=1)
df[["panelist_id", "verbatim"]].head(3)


,panelist_id,verbatim
0,67b0d135d362f5886c2e5cfe,"J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'..."
1,67b0d12ed362f5886c2e5b9c,Hmm... difficile. J'ai bien aimé les deux pour des raisons différentes. Mais je vais dire Bouygues. Juste pour l'ori...
2,67b0d12bd362f5886c2e5b09,"J'ai trouvé la pub Bouygues très drôle, mais je crois que je préfère celle d'Orange. Elle est plus simple, elle me p..."


In [4]:
SCHEMA_SCORES = {
    "informativeness": "Rate how informative, concrete, and factually useful each brand's ad feels.",
    "expressivity": "Rate how expressive, emotional, vivid, and affectively engaging each brand's ad feels.",
}

BRANDS = ["Bouygues", "Orange"]
ALL_DIMS = list(SCHEMA_SCORES.keys())

DIM_LABELS = {
    "informativeness": "Informativeness",
    "expressivity": "Expressivity",
}


In [5]:
_schema_lines = "\n".join(
    f"- {dim}: {rule}" for dim, rule in SCHEMA_SCORES.items()
)
_fields_lines = "\n".join(
    f'  "{dim}": {{"Bouygues": <0-10>, "Orange": <0-10>}},'
    for dim in SCHEMA_SCORES
)

SYSTEM_PROMPT = f"""Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panélistes ayant regardé deux publicités télévisées :
- Bouygues Telecom
- Orange

Pour chaque verbatim, attribue un score de 0 à 10 à chaque marque sur chaque dimension :
0 = pas du tout / absent, 10 = extrêmement fort / dominant.

DIMENSIONS :
{_schema_lines}

RÈGLES STRICTES :
1. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après.
2. Chaque score est un entier entre 0 et 10.
3. Ajoute un champ "reasoning" très court.
4. Ajoute un champ "confidence" entre 0.0 et 1.0.

FORMAT DE SORTIE :
{{
{_fields_lines}
  "reasoning": "explication courte",
  "confidence": 0.95
}}"""

FEW_SHOTS = [
    {
        "verbatim": "La pub Bouygues est plus claire sur le problème du wifi dans la maison et montre concrètement la solution. Celle d'Orange est plus sobre mais me touche moins.",
        "label": {
            "informativeness": {"Bouygues": 8, "Orange": 6},
            "expressivity": {"Bouygues": 7, "Orange": 4},
        },
    },
    {
        "verbatim": "Orange me paraît plus informative parce que le message sur la fiabilité est plus direct. Bouygues est plus théâtrale, plus drôle, plus vivante.",
        "label": {
            "informativeness": {"Bouygues": 6, "Orange": 9},
            "expressivity": {"Bouygues": 9, "Orange": 5},
        },
    },
    {
        "verbatim": "Bouygues est très expressive et mémorable, mais Orange explique mieux ce qu'elle promet vraiment.",
        "label": {
            "informativeness": {"Bouygues": 5, "Orange": 8},
            "expressivity": {"Bouygues": 9, "Orange": 4},
        },
    },
]


In [6]:
def extract_json(text):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group()) if m else None
    except Exception:
        return None

CONFIDENCE_THRESHOLD = 0.7

def build_user_prompt(verbatim):
    prompt = "EXEMPLES ANNOTÉS :\n"
    for ex in FEW_SHOTS:
        prompt += f"Verbatim: {ex['verbatim']}\n"
        prompt += f"Annotation: {json.dumps(ex['label'], ensure_ascii=False)}\n---\n"
    prompt += f"À ANNOTER :\n{verbatim}"
    return prompt

def validate_scores(parsed):
    for dim in SCHEMA_SCORES:
        if dim not in parsed:
            parsed[dim] = {"Bouygues": -1, "Orange": -1}
        else:
            for brand in BRANDS:
                val = parsed[dim].get(brand, -1)
                try:
                    parsed[dim][brand] = max(0, min(10, int(val)))
                except Exception:
                    parsed[dim][brand] = -1
    return parsed

def annotate(verbatim):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(verbatim)},
    ]
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": "mistral:7b", "messages": messages, "stream": False, "options": {"temperature": 0.1}},
        timeout=120,
    )
    generated = resp.json()["message"]["content"]
    parsed = extract_json(generated)
    if parsed is None:
        out = {dim: {"Bouygues": -1, "Orange": -1} for dim in SCHEMA_SCORES}
        out["reasoning"] = generated[:200]
        out["confidence"] = 0.0
        out["raw_output"] = generated
        return out
    parsed = validate_scores(parsed)
    confidence = float(parsed.get("confidence", 0.0))
    for dim in SCHEMA_SCORES:
        parsed[f"{dim}_flagged"] = confidence < CONFIDENCE_THRESHOLD
    parsed["raw_output"] = generated
    return parsed


In [7]:
test = annotate("Bouygues est plus expressive et plus marquante, mais Orange me paraît plus informative parce que sa promesse est beaucoup plus claire.")
test


{'informativeness': {'Bouygues': 6, 'Orange': 8},
 'expressivity': {'Bouygues': 9, 'Orange': 5},
 'reasoning': "Bouygues utilise une approche plus théâtrale et expressive pour captiver l'attention du public, tandis que Orange se concentre sur la clarité de sa promesse.",
 'confidence': 0.9,
 'informativeness_flagged': False,
 'expressivity_flagged': False,
 'raw_output': ' {\n  "informativeness": {"Bouygues": 6, "Orange": 8},\n  "expressivity": {"Bouygues": 9, "Orange": 5},\n  "reasoning": "Bouygues utilise une approche plus théâtrale et expressive pour captiver l\'attention du public, tandis que Orange se concentre sur la clarité de sa promesse.",\n  "confidence": 0.9\n}'}

In [8]:
PILOT_N = 30
pilot_df = df.sample(n=min(PILOT_N, len(df)), random_state=42).copy().reset_index(drop=True)

pilot_results = []
for i, row in pilot_df.iterrows():
    t0 = time.time()
    result = annotate(row["verbatim"])
    result["panelist_id"] = row["panelist_id"]
    pilot_results.append(result)
    elapsed = time.time() - t0
    print(
        f"{i+1:3d}/{len(pilot_df)} {elapsed:.1f}s | "
        + " ".join(f"{d}={result[d]['Bouygues']}v{result[d]['Orange']}" for d in ALL_DIMS)
        + f" | conf={result.get('confidence','?')}"
    )

pilot_results = pd.DataFrame(pilot_results)
pilot_results.to_csv("pilot_annotations_informativeness_expressivity.csv", index=False)
pilot_results.head()


  1/30 5.4s | informativeness=6v9 expressivity=7v8 | conf=0.95
  2/30 6.6s | informativeness=7v9 expressivity=6v8 | conf=0.95
  3/30 7.3s | informativeness=6v8 expressivity=7v9 | conf=0.95
  4/30 6.3s | informativeness=6v9 expressivity=5v8 | conf=0.95
  5/30 6.5s | informativeness=6v8 expressivity=7v9 | conf=0.85
  6/30 6.7s | informativeness=7v9 expressivity=8v6 | conf=0.95
  7/30 6.9s | informativeness=6v8 expressivity=7v9 | conf=0.95
  8/30 6.6s | informativeness=7v8 expressivity=6v9 | conf=0.95
  9/30 6.2s | informativeness=6v9 expressivity=7v8 | conf=0.95
 10/30 7.2s | informativeness=7v8 expressivity=6v9 | conf=0.95
 11/30 7.0s | informativeness=9v7 expressivity=10v5 | conf=0.95
 12/30 5.8s | informativeness=6v8 expressivity=7v6 | conf=0.95
 13/30 5.8s | informativeness=7v9 expressivity=8v6 | conf=0.95
 14/30 8.1s | informativeness=7v8 expressivity=9v6 | conf=0.95
 15/30 6.3s | informativeness=5v8 expressivity=9v7 | conf=0.95
 16/30 7.4s | informativeness=6v8 expressivity=7v9 | c

,informativeness,expressivity,reasoning,confidence,informativeness_flagged,expressivity_flagged,raw_output,panelist_id
0,"{'Bouygues': 6, 'Orange': 9}","{'Bouygues': 7, 'Orange': 8}",The Orange ad is more informative as it focuses on the practical benefits and real-life situations. The Bouygues ad ...,0.95,False,False,"{\n ""informativeness"": {""Bouygues"": 6, ""Orange"": 9},\n ""expressivity"": {""Bouygues"": 7, ""Orange"": 8},\n ""reasonin...",67b0d131d362f5886c2e5c61
1,"{'Bouygues': 7, 'Orange': 9}","{'Bouygues': 6, 'Orange': 8}","La publicité d'Orange est plus claire et humaine, avec des situations reconnues par le public. La publicité de Bouyg...",0.95,False,False,"{\n ""informativeness"": {\n ""Bouygues"": 7,\n ""Orange"": 9\n },\n ""expressivity"": {\n ""Bouygues"": 6,\n ...",67b0d12fd362f5886c2e5bdd
2,"{'Bouygues': 6, 'Orange': 8}","{'Bouygues': 7, 'Orange': 9}","Le verbatim indique que l'ad de Bouygues est plus originale et mémorable, mais Orange est plus pertinente et touche ...",0.95,False,False,"{\n ""informativeness"": {""Bouygues"": 6, ""Orange"": 8},\n ""expressivity"": {""Bouygues"": 7, ""Orange"": 9},\n ""reasonin...",67b0d125d362f5886c2e59f3
3,"{'Bouygues': 6, 'Orange': 9}","{'Bouygues': 5, 'Orange': 8}",La publicité d'Orange est plus informatrice en offrant des situations concrètes où une bonne connexion est indispens...,0.95,False,False,"{\n ""informativeness"": {""Bouygues"": 6, ""Orange"": 9},\n ""expressivity"": {""Bouygues"": 5, ""Orange"": 8},\n ""reasonin...",67b0d12dd362f5886c2e5b91
4,"{'Bouygues': 6, 'Orange': 8}","{'Bouygues': 7, 'Orange': 9}","La publicité d'Orange est plus claire et concrète dans sa présentation du produit, tandis que celle de Bouygues est ...",0.85,False,False,"{\n ""informativeness"": {\n ""Bouygues"": 6,\n ""Orange"": 8\n },\n ""expressivity"": {\n ""Bouygues"": 7,\n ...",67b0d125d362f5886c2e59ff


In [9]:
CHECKPOINT_EVERY = 50
checkpoint_path = "annotations_informativeness_expressivity_checkpoint.csv"
all_results = []
run_start = time.time()
total = len(df)

for i, row in df.iterrows():
    t0 = time.time()
    result = annotate(row["verbatim"])
    result["panelist_id"] = row["panelist_id"]
    all_results.append(result)
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)
        done = i + 1
        avg = (time.time() - run_start) / done
        eta = avg * (total - done) / 60
        print(f"{done}/{total} | checkpoint saved | {avg:.1f}s/sample | ETA {eta:.0f} min")

annotations = pd.DataFrame(all_results)
annotations.to_csv("preference_annotations_informativeness_expressivity.csv", index=False)
annotations.head()


50/800 | checkpoint saved | 6.3s/sample | ETA 79 min
100/800 | checkpoint saved | 6.5s/sample | ETA 75 min
150/800 | checkpoint saved | 6.5s/sample | ETA 70 min
200/800 | checkpoint saved | 6.5s/sample | ETA 65 min
250/800 | checkpoint saved | 6.5s/sample | ETA 60 min
300/800 | checkpoint saved | 6.5s/sample | ETA 54 min
350/800 | checkpoint saved | 6.6s/sample | ETA 49 min
400/800 | checkpoint saved | 6.6s/sample | ETA 44 min
450/800 | checkpoint saved | 6.6s/sample | ETA 38 min
500/800 | checkpoint saved | 6.6s/sample | ETA 33 min
550/800 | checkpoint saved | 6.6s/sample | ETA 27 min
600/800 | checkpoint saved | 6.6s/sample | ETA 22 min
650/800 | checkpoint saved | 6.6s/sample | ETA 16 min
700/800 | checkpoint saved | 6.5s/sample | ETA 11 min
750/800 | checkpoint saved | 6.6s/sample | ETA 5 min
800/800 | checkpoint saved | 6.5s/sample | ETA 0 min


,informativeness,expressivity,reasoning,confidence,informativeness_flagged,expressivity_flagged,raw_output,panelist_id
0,"{'Bouygues': 8, 'Orange': 7}","{'Bouygues': 9, 'Orange': 6}","La pub Bouygues est plus marrante et originale, avec un concept absurde qui la rend mémorable. La pub Orange est eff...",0.95,False,False,"{\n ""informativeness"": {""Bouygues"": 8, ""Orange"": 7},\n ""expressivity"": {""Bouygues"": 9, ""Orange"": 6},\n ""reasonin...",67b0d135d362f5886c2e5cfe
1,"{'Bouygues': 7, 'Orange': 8}","{'Bouygues': 9, 'Orange': 6}","Bouygues' ad is more expressive and memorable due to its creative concept of a 'WiFi police', which is easy to remem...",0.95,False,False,"{\n ""informativeness"": {""Bouygues"": 7, ""Orange"": 8},\n ""expressivity"": {""Bouygues"": 9, ""Orange"": 6},\n ""reasonin...",67b0d12ed362f5886c2e5b9c
2,"{'Bouygues': 7, 'Orange': 8}","{'Bouygues': 9, 'Orange': 6}","The Bouygues ad is more expressive and memorable due to its original investigation theme, while Orange's ad is more ...",0.85,False,False,"{\n ""informativeness"": {""Bouygues"": 7, ""Orange"": 8},\n ""expressivity"": {""Bouygues"": 9, ""Orange"": 6},\n ""reasonin...",67b0d12bd362f5886c2e5b09
3,"{'Bouygues': 6, 'Orange': 8}","{'Bouygues': 7, 'Orange': 5}","L'annonceur d'Orange a présenté une image plus claire et directe de la fiabilité de son service, tandis que l'annonc...",0.95,False,False,"{\n ""informativeness"": {\n ""Bouygues"": 6,\n ""Orange"": 8\n },\n ""expressivity"": {\n ""Bouygues"": 7,\n ...",67b0d127d362f5886c2e5a4a
4,"{'Bouygues': 7, 'Orange': 8}","{'Bouygues': 9, 'Orange': 6}","La pub Bouygues est plus créative et amusante, ce qui la rend mémorable. Elle utilise un concept unique ('scène de c...",0.95,False,False,"{\n ""informativeness"": {""Bouygues"": 7, ""Orange"": 8},\n ""expressivity"": {""Bouygues"": 9, ""Orange"": 6},\n ""reasonin...",67b0d12cd362f5886c2e5b4a


In [10]:
final = pd.read_csv("preference_annotations_informativeness_expressivity.csv")

for dim in ALL_DIMS:
    for brand in BRANDS:
        col = f"{dim}_{brand.lower()}"
        if isinstance(final[dim].iloc[0], str):
            final[col] = final[dim].apply(
                lambda x: json.loads(x.replace("'", '"')).get(brand, np.nan)
                if pd.notna(x) else np.nan
            )
        else:
            final[col] = final[dim].apply(lambda x: x.get(brand, np.nan) if isinstance(x, dict) else np.nan)

score_cols = [f"{dim}_{b.lower()}" for dim in ALL_DIMS for b in BRANDS]
cols_to_merge = ["panelist_id"] + score_cols + ["confidence", "reasoning"]
panel_merged = panel.merge(final[cols_to_merge], on="panelist_id", how="left")
panel_merged.to_csv("interviews_with_informativeness_expressivity_scores.csv", index=False)
print(panel_merged.shape)
panel_merged[["panelist_id"] + score_cols].head()


(800, 54)


,panelist_id,informativeness_bouygues,informativeness_orange,expressivity_bouygues,expressivity_orange
0,67b0d135d362f5886c2e5cfe,8,7,9,6
1,67b0d12ed362f5886c2e5b9c,7,8,9,6
2,67b0d12bd362f5886c2e5b09,7,8,9,6
3,67b0d127d362f5886c2e5a4a,6,8,7,5
4,67b0d12cd362f5886c2e5b4a,7,8,9,6


In [11]:
summary = pd.DataFrame({
    "Dimension": [DIM_LABELS[d] for d in ALL_DIMS],
    "Bouygues": [round(final[f"{d}_bouygues"].mean(), 2) for d in ALL_DIMS],
    "Orange": [round(final[f"{d}_orange"].mean(), 2) for d in ALL_DIMS],
}).set_index("Dimension")

summary.to_csv("informativeness_expressivity_summary.csv")
summary


,Bouygues,Orange
Dimension,,
Informativeness,6.72,8.09
Expressivity,7.70,6.89


In [12]:
BRAND_COLORS = {"Bouygues": "#0055A4", "Orange": "#FF6600"}
MARGINS = dict(t=80, b=120, l=60, r=20)
LEGEND_BELOW = dict(orientation="h", yanchor="top", y=-0.18, xanchor="center", x=0.5)

DEMOGRAPHICS = {
    "gender": "Gender",
    "age_group": "Age",
    "csp": "CSP",
    "income_level": "Income level",
    "education": "Education",
    "location.city_size": "City size",
}


In [13]:
if "age" in panel_merged.columns:
    panel_merged["age_group"] = pd.cut(
        panel_merged["age"],
        bins=[17, 24, 34, 44, 54, 64, 120],
        labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    )

available_demographics = {k: v for k, v in DEMOGRAPHICS.items() if k in panel_merged.columns}
available_demographics


{'gender': 'Gender',
 'age_group': 'Age',
 'csp': 'CSP',
 'income_level': 'Income level',
 'education': 'Education',
 'location.city_size': 'City size'}

In [14]:
def prep_demo_col(df, col, min_count=10, max_levels=12):
    out = df.copy()
    out = out[out[col].notna()].copy()
    out["demo"] = out[col].astype(str)
    out = out[out["demo"] != "nan"].copy()
    counts = out["demo"].value_counts()
    keep = counts[counts >= min_count].index.tolist()
    if len(keep) == 0:
        keep = counts.index.tolist()
    out["demo"] = np.where(out["demo"].isin(keep), out["demo"], "Other")
    counts2 = out["demo"].value_counts()
    top_levels = counts2.index.tolist()[:max_levels]
    out = out[out["demo"].isin(top_levels)].copy()
    return out


In [15]:
def make_score_facet_plot(df, dim, demo_col, demo_label):
    plot_df = prep_demo_col(df, demo_col)
    score_cols_dim = [f"{dim}_{b.lower()}" for b in BRANDS]
    long = plot_df[["demo"] + score_cols_dim].melt(
        id_vars="demo",
        value_vars=score_cols_dim,
        var_name="brand_col",
        value_name="score",
    )
    long["brand"] = long["brand_col"].str.replace(f"{dim}_", "", regex=False).str.capitalize()
    long["brand"] = long["brand"].replace({"Bouygues": "Bouygues", "Orange": "Orange"})

    grouped = (
        long.groupby(["demo", "brand"])["score"]
        .agg(mean_score="mean", sem=lambda x: x.sem())
        .reset_index()
    )
    grouped.rename(columns={"demo": demo_label}, inplace=True)

    n_levels = grouped[demo_label].nunique()
    n_cols = min(3, max(1, n_levels))
    n_rows = math.ceil(n_levels / n_cols)

    fig = px.bar(
        grouped,
        x="brand",
        y="mean_score",
        color="brand",
        facet_col=demo_label,
        facet_col_wrap=n_cols,
        error_y="sem",
        color_discrete_map=BRAND_COLORS,
        title=f"{DIM_LABELS[dim]} by {demo_label}",
        range_y=[0, 10],
    )
    fig.update_layout(
        showlegend=False,
        margin=dict(t=80, b=50, l=50, r=20),
        height=max(450, 260 * n_rows),
    )
    fig.update_yaxes(title_text="Mean score (0–10)", matches=None)
    fig.update_xaxes(title_text="")
    return fig


In [16]:
fig = go.Figure()
for brand in BRANDS:
    fig.add_trace(go.Bar(
        name=brand,
        x=[DIM_LABELS[d] for d in ALL_DIMS],
        y=[round(final[f"{d}_{brand.lower()}"].mean(), 2) for d in ALL_DIMS],
        marker_color=BRAND_COLORS[brand],
    ))

fig.update_layout(
    barmode="group",
    title=dict(text="Mean scores across all dimensions", x=0, xanchor="left"),
    legend=LEGEND_BELOW,
    margin=MARGINS,
    yaxis=dict(title="Mean score (0–10)", range=[0, 10]),
    xaxis=dict(title=""),
)
fig.show()


In [17]:
for dim in ALL_DIMS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_score_facet_plot(panel_merged, dim, demo_col, demo_label)
        fig.show()


In [18]:
def slugify(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^\w]+", "_", s).strip("_").lower()

output_dir = Path("informativeness_expressivity_facet_charts")
output_dir.mkdir(exist_ok=True)
saved_files = []

for dim in ALL_DIMS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_score_facet_plot(panel_merged, dim, demo_col, demo_label)
        plot_df = prep_demo_col(panel_merged, demo_col)
        n_levels = plot_df["demo"].nunique() if len(plot_df) else 1
        n_cols = min(3, max(1, n_levels))
        n_rows = max(1, math.ceil(n_levels / n_cols))
        fig.update_layout(
            width=2400,
            height=max(1000, 420 * n_rows),
            margin=dict(t=100, b=80, l=70, r=40),
            title_x=0.01,
            font=dict(size=16),
        )
        fig.update_annotations(font_size=15)
        out = output_dir / f"{slugify(dim)}_by_{slugify(demo_col)}.png"
        fig.write_image(str(out), width=2400, height=max(1000, 420 * n_rows), scale=2)
        saved_files.append(str(out))

len(saved_files)


12